In [1]:
%load_ext autoreload
%autoreload 2
import scMPRAforge as scm
import pandas as pd


2025-08-27 15:00:27.700346: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-27 15:00:27.704465: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/code-server/4.91.1/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

## Test wald test

In [2]:

#create dask cluster
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(n_workers=10)
client = Client(cluster)

path="/gpfs/gibbs/pi/reilly/tabula_data/shendure"
name="ortho_primordial"
precomp_name = 'ortho_primordial_wald_precomp'

import os
if os.path.isdir(path+"/"+precomp_name):
    print("[+]PreComp Model found. Loading...")
    primordial=scm.ortho.load(client,path,precomp_name)
    shendure=primordial.training_data
elif os.path.isdir(path+"/"+name):
    print("[+] Model found. Loading...")
    primordial=scm.ortho.load(client,path,name)
    shendure=primordial.training_data
    primordial.precompute_wald(client)
    primordial.save(path, precomp_name) 


[+]PreComp Model found. Loading...


In [3]:
# by-cell-type: test many CREs in NeuroectodermBrain vs the 'reference' negative control
hs_ct = scm.make_by_celltype_hypotheses(
    comparison_cell_type="NeuroectodermBrain",
    counts=shendure,
    comparison_cres="all",          # or a list like ["CRE1","CRE2",...]
    reference_cre="reference",      # this is how you labeled minP/noP
    meta="emvar_screen"
)

# by-CRE: test CRE123 across all cell types vs the baseline cell type
hs_cre = scm.make_by_cre_hypotheses(
    comparison_cre="all",
    counts=shendure,
    comparison_cell_types="all",    # or a list
    reference_cell_type="reference",   # will default to counts.reference_cell_type if set
    meta="cell_specificity"
)

# all CREs within each cell type, vs the 'reference' negative control
hs_all_ct = scm.make_all_by_celltype_hypotheses(
    counts=shendure,
    reference_cre="reference",
    meta="emvar_screen",
)

# all cell types for each CRE, vs the dataset’s baseline cell type
hs_all_cre = scm.make_all_by_cre_hypotheses(
    counts=shendure,
    reference_cell_type="reference",  # will be normalized to 'reference'
    meta="cell_specificity",
)




In [4]:
runner = scm.HypothesisTester("wald")
wald_by_ct  = runner.run(hs_all_ct, primordial, client).to_dataframe()
wald_by_cre = runner.run(hs_all_cre, primordial, client).to_dataframe()

In [5]:
runner = scm.HypothesisTester("mwu", alternative="two-sided")  # can override defaults here
mwu_res = runner.run(hs_all_ct, shendure, client=client).to_dataframe()

In [6]:
mwu_res.fold_change.describe()

count    1451.000000
mean       19.216108
std       112.569072
min         0.118643
25%         0.778496
50%         1.612333
75%         4.563912
max      1751.045275
Name: fold_change, dtype: float64

In [7]:
mwu_res

,comparison_CRE,comparison_cell_type,reference_CRE,reference_cell_type,meta,test_statistic,p_value,fold_change,flattened,ref_mean,comp_mean,test_type,bh_p
0,Bend5_chr4_8175,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,25246.5,2.364681e-58,33.003656,True,0.024053,1.113858,mwu,1.531764e-57
1,Cdk5r1_chr11_12559,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,53572.0,2.047382e-16,6.230655,True,0.024053,0.202169,mwu,7.483002e-16
2,Col1a1_chr11_15322,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,14121.0,3.956297e-04,3.430988,True,0.024053,0.106834,mwu,8.073963e-04
3,Col1a2_chr6_77,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,25934.5,5.874620e-02,1.580133,True,0.024053,0.043807,mwu,8.706920e-02
4,Igfbp4_chr11_16711,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,15121.0,8.211205e-03,2.435112,True,0.024053,0.072922,mwu,1.437208e-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1446,Txndc12_chr4_7973,reference,reference,reference,emvar_screen,3878405.0,0.000000e+00,14.526987,True,0.107775,1.700917,mwu,0.000000e+00
1447,Lama1_chr17_7787,reference,reference,reference,emvar_screen,902038.5,1.062349e-04,0.277101,True,0.107775,0.022636,mwu,2.293853e-04
1448,Epas1_chr17_10064,reference,reference,reference,emvar_screen,1047294.5,3.689841e-03,0.494174,True,0.107775,0.048201,mwu,6.751526e-03
1449,Btg1_chr10_9578,reference,reference,reference,emvar_screen,1835252.0,0.000000e+00,36.953790,True,0.107775,4.342235,mwu,0.000000e+00
